- Don't let perfection be the enemy of good
- Network is your net worth


In [ ]:
import pandas as pd

df = pd.read_csv("./data/bpm_data_to_merge.csv", dtype={56: str})
injuries = pd.read_csv("./data/injuries_code_classified.csv")
df

/Users/ryanschwartz/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,game_id,season,season_type,game_date,game_date_time,athlete_id,athlete_display_name,team_id,team_name,team_location,...,average_bpm_contribution,avg*mins,cumulative_average_bpm_contribution,final_average_bpm_contribution,cumulative_average_z_score_player_season,calculated_position,bpm,season_bpm,season_bpm_zscore,cumulative_adjusted_bpm_zscore
0,211030019,2002,2,2001-10-30,2001-10-31T00:30:00Z,353.0,Troy Hudson,19,Magic,Orlando,...,-2.790514,-53.019766,-2.790514,3.514053,-1.761281,1.860991,4.799162,-1.186313,-0.576446,-0.663464
1,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,376.0,Bobby Jackson,23,Kings,Sacramento,...,0.176965,3.362337,0.176965,6.827172,-0.982548,2.279120,-2.176561,1.826902,0.558563,-0.777526
2,211030023,2002,2,2001-10-30,2001-10-31T00:00:00Z,862.0,Hedo Turkoglu,23,Kings,Sacramento,...,2.250545,51.762538,2.250545,5.387655,-0.438395,2.752464,-2.791259,1.344887,0.376999,-0.511514
3,211030026,2002,2,2001-10-30,2001-10-31T00:00:00Z,517.0,Anthony Mason,15,Bucks,Milwaukee,...,5.431422,228.119737,5.431422,4.816615,0.396337,3.074812,0.174913,-0.346934,-0.260271,0.133507
4,211030026,2002,2,2001-10-30,2001-10-31T00:00:00Z,138.0,Sam Cassell,15,Bucks,Milwaukee,...,17.500972,752.541809,17.500972,8.467723,3.563654,1.759043,10.581040,2.886000,0.957502,1.264824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493476,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,3033.0,P.J. Tucker,18,Knicks,New York,...,0.289925,8.407836,0.252096,0.252096,-1.186355,3.010192,-1.649800,-4.974789,NaN,0.707107
493477,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,4431679.0,Precious Achiuwa,18,Knicks,New York,...,2.033320,67.099572,5.644024,5.644024,0.000030,2.972519,-1.372852,-1.497798,-0.693775,-0.437459
493478,401705754,2025,2,2025-04-13,2025-04-13T17:00:00Z,3064230.0,Cameron Payne,18,Knicks,New York,...,12.666785,430.670702,7.800168,7.811829,0.474445,2.037766,8.502112,0.026893,NaN,0.417675
493479,401705753,2025,2,2025-04-13,2025-04-13T17:00:00Z,4576085.0,JD Davison,2,Celtics,Boston,...,8.125943,138.141030,-2.435398,-2.435398,-1.777684,2.080286,0.047324,-7.094774,NaN,0.707107


In [2]:
df = df.query("season >= 2010 and season <= 2024")

In [3]:
df.rename(columns={"athlete_display_name": "player"}, inplace=True)

/var/folders/v7/1_gf4cd15qgd59psn6gym7_r0000gn/T/ipykernel_14095/4179684171.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"athlete_display_name": "player"}, inplace=True)


In [4]:
# drop times there were two players with the same name in a given season, who had different ids

# Find all player names that have more than one unique athlete_id
name_id_counts = df.groupby("player")["athlete_id"].nunique()
duplicate_names = name_id_counts[name_id_counts > 1].index

# Drop all rows for players with duplicate names (multiple athlete_ids)
df = df[~df["player"].isin(duplicate_names)]

In [5]:
injuries

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
0,2001-07-01,76ers,Craig Claxton,NaN,activated from IL,NaN
1,2001-07-01,Clippers,Corey Maggette,NaN,activated from IL,NaN
2,2001-07-01,Clippers,Earl Boykins,NaN,activated from IL,NaN
3,2001-07-01,Grizzlies,Doug West,NaN,activated from IL,NaN
4,2001-07-01,Knicks,Luc Longley,NaN,activated from IL,NaN
...,...,...,...,...,...,...
32815,2023-04-16,Clippers,Marcus Morris,NaN,activated from IL,NaN
32816,2023-04-16,Grizzlies,Dillon Brooks,NaN,activated from IL,NaN
32817,2023-04-16,Grizzlies,Ja Morant,NaN,activated from IL,NaN
32818,2023-04-16,Grizzlies,Jaren Jackson Jr.,NaN,activated from IL,NaN


This works, don't know why


## Creating game count


In [6]:
# Create a unique identifier for each team-game
df["team_game_key"] = (
    df["season"].astype(str)
    + "_"
    + df["team_id"].astype(str)
    + "_"
    + df["game_id"].astype(str)
)

# Drop duplicates to get unique team-games only
unique_team_games = df[
    ["team_game_key", "season", "team_id", "game_date"]
].drop_duplicates()

# Sort those for proper counting
unique_team_games = unique_team_games.sort_values(["season", "team_id", "game_date"])

# Use a dictionary to track game_count
game_count_dict = {}

counter = {}
for row in unique_team_games.itertuples(index=False):
    key = row.team_game_key
    team_ = row.team_id
    season = row.season
    group_key = (season, team_)

    if group_key not in counter:
        counter[group_key] = 1
    else:
        counter[group_key] += 1

    game_count_dict[key] = counter[group_key]

# Map back to the full DataFrame (very efficient)
df["game_count"] = df["team_game_key"].map(game_count_dict)

# Optionally drop the helper column
df.drop(columns="team_game_key", inplace=True)

## Creating periods


In [7]:
# # Assume df is your main DataFrame with player games
# # Assume injuries is your DataFrame with 'Acquired', 'Relinquished', 'Date'

# # 1. Get all "come back from injury" events
# comebacks = injuries.loc[~injuries["Acquired"].isna(), ["Acquired", "Date"]].copy()
# comebacks = comebacks.rename(columns={"Acquired": "player", "Date": "event_date"})
# comebacks["event_type"] = "comeback"
# comebacks

In [8]:
# # 2. Get all season starts for each player
# season_starts = (
#     df.sort_values(["athlete_display_name", "season", "game_date"])
#     .groupby(["athlete_display_name", "season"])
#     .first()
#     .reset_index()[["athlete_display_name", "season", "game_date"]]
# )
# season_starts = season_starts.rename(columns={"game_date": "event_date"})
# season_starts["event_type"] = "season_start"
# season_starts

In [9]:
# # 3. Combine events
# events = pd.concat([season_starts, comebacks], ignore_index=True)
# events["event_date"] = pd.to_datetime(events["event_date"])
# events = events.sort_values(["athlete_display_name", "event_date"])

# # 4. Assign period numbers
# events["period"] = events.groupby("athlete_display_name").cumcount() + 1
# events

In [10]:
# # 5. Merge periods back to df
# df["game_date"] = pd.to_datetime(df["game_date"])
# df = df.sort_values(["athlete_display_name", "game_date"])


# def assign_period(row, events):
#     player_events = events[
#         events["athlete_display_name"] == row["athlete_display_name"]
#     ]
#     # Find the last event before or on this game_date
#     prior_events = player_events[player_events["event_date"] <= row["game_date"]]
#     if not prior_events.empty:
#         return prior_events.iloc[-1]["period"]
#     else:
#         return 1  # Default to 1 if no event found


# df["period"] = df.apply(lambda row: assign_period(row, events), axis=1)

In [11]:
injuries

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
0,2001-07-01,76ers,Craig Claxton,NaN,activated from IL,NaN
1,2001-07-01,Clippers,Corey Maggette,NaN,activated from IL,NaN
2,2001-07-01,Clippers,Earl Boykins,NaN,activated from IL,NaN
3,2001-07-01,Grizzlies,Doug West,NaN,activated from IL,NaN
4,2001-07-01,Knicks,Luc Longley,NaN,activated from IL,NaN
...,...,...,...,...,...,...
32815,2023-04-16,Clippers,Marcus Morris,NaN,activated from IL,NaN
32816,2023-04-16,Grizzlies,Dillon Brooks,NaN,activated from IL,NaN
32817,2023-04-16,Grizzlies,Ja Morant,NaN,activated from IL,NaN
32818,2023-04-16,Grizzlies,Jaren Jackson Jr.,NaN,activated from IL,NaN


In [12]:
# # Ensure Date is datetime for sorting
# injuries["Date"] = pd.to_datetime(injuries["Date"])


# # For each Acquired event, find the most recent Relinquished event for that player before this Date
# def get_previous_injury(row):
#     if pd.isna(row["Acquired"]):
#         return row["Injury_Category"]
#     prev = injuries[
#         (injuries["Relinquished"] == row["Acquired"])
#         & (pd.to_datetime(injuries["Date"]) < row["Date"])
#     ].sort_values("Date", ascending=False)
#     if not prev.empty:
#         return prev.iloc[0]["Injury_Category"]
#     return row["Injury_Category"]


# injuries["Injury_Category"] = injuries.apply(get_previous_injury, axis=1)

In [13]:
injuries

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
0,2001-07-01,76ers,Craig Claxton,NaN,activated from IL,NaN
1,2001-07-01,Clippers,Corey Maggette,NaN,activated from IL,NaN
2,2001-07-01,Clippers,Earl Boykins,NaN,activated from IL,NaN
3,2001-07-01,Grizzlies,Doug West,NaN,activated from IL,NaN
4,2001-07-01,Knicks,Luc Longley,NaN,activated from IL,NaN
...,...,...,...,...,...,...
32815,2023-04-16,Clippers,Marcus Morris,NaN,activated from IL,NaN
32816,2023-04-16,Grizzlies,Dillon Brooks,NaN,activated from IL,NaN
32817,2023-04-16,Grizzlies,Ja Morant,NaN,activated from IL,NaN
32818,2023-04-16,Grizzlies,Jaren Jackson Jr.,NaN,activated from IL,NaN


In [14]:
# injuries.to_csv("./data/injuries_code_classified_all_categories.csv", index=False)

In [15]:
injuries = pd.read_csv("./data/injuries_code_classified_all_categories.csv")
injuries

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
0,2001-07-01,76ers,Craig Claxton,NaN,activated from IL,NaN
1,2001-07-01,Clippers,Corey Maggette,NaN,activated from IL,NaN
2,2001-07-01,Clippers,Earl Boykins,NaN,activated from IL,NaN
3,2001-07-01,Grizzlies,Doug West,NaN,activated from IL,NaN
4,2001-07-01,Knicks,Luc Longley,NaN,activated from IL,NaN
...,...,...,...,...,...,...
32815,2023-04-16,Clippers,Marcus Morris,NaN,activated from IL,Back
32816,2023-04-16,Grizzlies,Dillon Brooks,NaN,activated from IL,Groin
32817,2023-04-16,Grizzlies,Ja Morant,NaN,activated from IL,Hand
32818,2023-04-16,Grizzlies,Jaren Jackson Jr.,NaN,activated from IL,Elbow


In [16]:
injuries.query(
    "Acquired == 'Klay Thompson' or Relinquished == 'Klay Thompson'"
).sort_values("Date")

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
17060,2015-03-18,Warriors,NaN,Klay Thompson,placed on IL with sprained right ankle,Ankle
17098,2015-03-23,Warriors,Klay Thompson,NaN,activated from IL,Ankle
19772,2017-01-10,Warriors,NaN,Klay Thompson,placed on IL for rest,Other
19793,2017-01-12,Warriors,Klay Thompson,NaN,activated from IL,Other
21969,2018-01-10,Warriors,NaN,Klay Thompson,placed on IL,Other
21986,2018-01-12,Warriors,Klay Thompson,NaN,activated from IL,Other
22645,2018-03-14,Warriors,NaN,Klay Thompson,placed on IL with fractured right thumb,Finger
22903,2018-04-01,Warriors,Klay Thompson,NaN,activated from IL,Finger
24396,2019-03-05,Warriors,NaN,Klay Thompson,placed on IL with sore right knee,Knee (Minor)
24421,2019-03-08,Warriors,Klay Thompson,NaN,activated from IL,Knee (Minor)


In [17]:
# df = df.query("player == 'Klay Thompson'")

In [18]:
# # 1. Prepare comebacks DataFrame
# comebacks = injuries.loc[
#     ~injuries["Acquired"].isna(), ["Acquired", "Date", "Injury_Category"]
# ].copy()
# comebacks = comebacks.rename(
#     columns={"Acquired": "player", "Date": "event_date", "Injury_Category": "injury"}
# )
# comebacks["event_type"] = "comeback"

# # 2. Prepare season starts DataFrame, but only if first game_count == 1
# season_starts_all = (
#     df.sort_values(["player", "season", "game_date"])
#     .groupby(["player", "season"])
#     .first()
#     .reset_index()[["player", "season", "game_date", "game_count"]]
# )

# # Only keep season starts where game_count == 1
# season_starts = season_starts_all[season_starts_all["game_count"] == 1].copy()
# season_starts = season_starts.rename(columns={"game_date": "event_date"})
# season_starts["event_type"] = "season_start"
# season_starts["injury"] = None  # No injury for season start

# # 3. For first game in season where game_count > 1, set as comeback with previous injury
# late_starts = season_starts_all[season_starts_all["game_count"] > 1].copy()
# late_starts = late_starts.rename(columns={"game_date": "event_date"})
# late_starts["event_type"] = "comeback"
# # melt it so aquired and relinquished are in the same column
# # Sort to ensure chronological order
# # injuries = injuries.sort_values(["Relinquished", "Date"]).reset_index(drop=True)

# # # Identify when a new block of consecutive non-null 'Relinquished' values begins
# # injuries["is_new_injury"] = (
# #     injuries["Relinquished"].notna()
# #     & (injuries["Relinquished"] != injuries["Relinquished"].shift())  # player changed
# # ) | (
# #     injuries["Relinquished"].notna()
# #     & injuries["Relinquished"].eq(injuries["Relinquished"].shift())
# #     & injuries["Relinquished"].notna()
# #     & injuries["Relinquished"].shift().isna()  # first of a streak
# # )

# # # Filter to keep only those "first" injuries
# # filtered_injuries = injuries[injuries["is_new_injury"]].copy()


# # # Find previous injury for each late start
# # def get_prev_injury(row):
# #     player = row["player"]
# #     event_date = row["event_date"]
# #     prev_inj = filtered_injuries[
# #         (filtered_injuries["Relinquished"] == player)
# #         & (filtered_injuries["Date"] < event_date)
# #     ].sort_values(
# #         "Date", ascending=True
# #     )  # earliest first
# #     if not prev_inj.empty:
# #         return prev_inj.iloc[0]["Injury_Category"]
# #     return None

# injuries_melted = injuries.melt(
#     id_vars=["Date", "Injury_Category"],
#     value_vars=["Acquired", "Relinquished"],
#     var_name="type",
#     value_name="player",
# )


# def get_prev_injury(row):
#     player = row["player"]
#     event_date = pd.to_datetime(row["event_date"])

#     # Find all entries for this player
#     player_events = injuries_melted[injuries_melted["player"] == player].copy()
#     player_events["Date"] = pd.to_datetime(player_events["Date"])
#     player_events = player_events.sort_values("Date")

#     # Find relevant events before this date
#     before_events = player_events[player_events["Date"] < event_date]

#     if before_events.empty:
#         return None

#     # Find the last comeback (Acquired) before this event
#     last_comeback = before_events[before_events["type"] == "Acquired"]

#     if not last_comeback.empty:
#         last_comeback_date = last_comeback["Date"].max()
#         # Find all injuries after the last comeback and before this event
#         injuries_since_comeback = before_events[
#             (before_events["type"] == "Relinquished")
#             & (before_events["Date"] > last_comeback_date)
#         ]

#         if not injuries_since_comeback.empty:
#             # Return the first injury since the last comeback
#             return injuries_since_comeback.iloc[0]["Injury_Category"]

#     # If no comeback found or no injuries after comeback,
#     # just get the most recent injury before the event
#     recent_injuries = before_events[before_events["type"] == "Relinquished"]

#     if not recent_injuries.empty:
#         return recent_injuries.iloc[-1]["Injury_Category"]

#     return None


# late_starts["injury"] = late_starts.apply(get_prev_injury, axis=1)
# comebacks["injury"] = comebacks.apply(get_prev_injury, axis=1)  # Add this line
# late_starts["event_type"] = "comeback"

# # 4. Combine all events
# all_events = pd.concat([season_starts, comebacks, late_starts], ignore_index=True)
# all_events["event_date"] = pd.to_datetime(all_events["event_date"])
# all_events = all_events.sort_values(["player", "event_date"])
# all_events["period"] = all_events.groupby("player").cumcount() + 1

# # 5. Merge periods back to df, keeping event_type and injury
# df = df.sort_values(["player", "game_date"])


# def assign_period_info(row, events):
#     player_events = events[events["player"] == row["player"]]
#     prior_events = player_events[player_events["event_date"] <= row["game_date"]]
#     if not prior_events.empty:
#         last_event = prior_events.iloc[-1]
#         return pd.Series(
#             {
#                 "period": last_event["period"],
#                 "period_event_type": last_event["event_type"],
#                 "period_injury": last_event["injury"],
#             }
#         )
#     else:
#         return pd.Series(
#             {"period": 1, "period_event_type": None, "period_injury": None}
#         )


# df[["period", "period_event_type", "period_injury"]] = df.apply(
#     lambda row: assign_period_info(row, all_events), axis=1
# )

In [19]:
# df.to_csv("./data/df_with_periods.csv", index=False)

In [20]:
df = pd.read_csv("./data/df_with_periods.csv", dtype={56: str, 149: str})
df

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,cumulative_average_z_score_player_season,calculated_position,bpm,season_bpm,season_bpm_zscore,cumulative_adjusted_bpm_zscore,game_count,period,period_event_type,period_injury
0,401468968,2023,2,2023-02-06,2023-02-07T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,-4.127669,2.436696,-3.601447,-5.589116,NaN,-1.161583,55,1,comeback,NaN
1,401469036,2023,2,2023-02-15,2023-02-16T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,0.012540,2.436696,-2.333228,-5.589116,NaN,1.270587,60,1,comeback,NaN
2,401469365,2023,2,2023-04-07,2023-04-08T00:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,0.036306,2.436696,-1.202561,-5.589116,NaN,0.068767,81,1,comeback,NaN
3,401469379,2023,2,2023-04-09,2023-04-09T19:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,-0.231331,2.436696,-12.711441,-5.589116,NaN,-0.177771,82,1,comeback,NaN
4,401584915,2024,2,2023-12-02,2023-12-03T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,1.205269,2.311806,-6.855364,-4.487577,NaN,1.166410,19,2,comeback,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311343,310408014,2011,2,2011-04-08,2011-04-08T23:30:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,-0.150128,3.105684,0.300598,-0.012866,-0.134435,0.603101,79,10,comeback,Other
311344,310411001,2011,2,2011-04-11,2011-04-11T23:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,-0.165710,3.105684,-2.630987,-0.012866,-0.134435,-0.317412,81,10,comeback,Other
311345,310413028,2011,2,2011-04-13,2011-04-14T00:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,-0.161364,3.105684,3.706229,-0.012866,-0.134435,0.151461,82,10,comeback,Other
311346,401160820,2020,2,2019-11-16,2019-11-17T01:00:00Z,3137713.0,Zylan Cheatham,3,Pelicans,New Orleans,...,-3.393773,2.572996,-12.677736,-8.356102,NaN,-0.707107,12,1,comeback,NaN


We have combacks now. Just need period injury, which would tell us the injury they had in previous period?


In [21]:
df.rename(columns={"period_injury": "comeback_from_injury"}, inplace=True)
df.query("player == 'Klay Thompson' and season==2015").sort_values("game_date").tail(20)

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,cumulative_average_z_score_player_season,calculated_position,bpm,season_bpm,season_bpm_zscore,cumulative_adjusted_bpm_zscore,game_count,period,period_event_type,comeback_from_injury
179547,400579199,2015,2,2015-03-04,2015-03-05T03:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.741772,2.349774,2.882510,5.866904,2.08034,-0.282830,59,4,season_start,NaN
179548,400579214,2015,2,2015-03-06,2015-03-07T03:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.694718,2.349774,1.116985,5.866904,2.08034,-1.278473,60,4,season_start,NaN
179549,400579224,2015,2,2015-03-08,2015-03-08T19:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.707078,2.349774,6.789358,5.866904,2.08034,0.250979,61,4,season_start,NaN
179550,400579236,2015,2,2015-03-09,2015-03-10T02:00:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.711631,2.349774,9.801499,5.866904,2.08034,0.042615,62,4,season_start,NaN
179551,400579252,2015,2,2015-03-11,2015-03-12T02:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.730606,2.349774,9.278572,5.866904,2.08034,0.378337,63,4,season_start,NaN
179552,400579273,2015,2,2015-03-14,2015-03-15T02:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.769262,2.349774,17.313759,5.866904,2.08034,1.194873,65,4,season_start,NaN
179553,400579291,2015,2,2015-03-16,2015-03-17T02:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.760527,2.349774,1.746324,5.866904,2.08034,-0.268760,66,4,season_start,NaN
179554,400579345,2015,2,2015-03-23,2015-03-24T02:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.745998,2.349774,9.423507,5.866904,2.08034,-0.760302,70,5,comeback,Ankle
179555,400579351,2015,2,2015-03-24,2015-03-25T02:30:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.751417,2.349774,3.304294,5.866904,2.08034,0.128610,71,5,comeback,Ankle
179556,400579374,2015,2,2015-03-27,2015-03-28T00:00:00Z,6475.0,Klay Thompson,9,Warriors,Golden State,...,1.795198,2.349774,16.770088,5.866904,2.08034,1.387844,72,5,comeback,Ankle


In [22]:
# Find all player names that have more than one unique athlete_id
ambiguous_names = (
    df.groupby("player")["athlete_id"].nunique().reset_index().query("athlete_id > 1")
)

# Show all rows in df where this occurs

df = df.loc[~df["player"].isin(ambiguous_names["player"])]

# FIX THIS


In [23]:
# # 6. Collapse periods with <5 games into previous period
def collapse_short_periods(df, min_games=5):
    df = df.copy()
    # For each player/season, process periods
    for (player, season), group in df.groupby(["player", "season"]):
        period_counts = group["period"].value_counts().sort_index()
        periods = period_counts.index.tolist()
        counts = period_counts.values
        # Map for period reassignment
        period_map = {}
        prev_period = None
        for i, (p, count) in enumerate(zip(periods, counts)):
            if count < min_games and prev_period is not None:
                period_map[p] = prev_period
            else:
                period_map[p] = p
                prev_period = p
        mask = (df["player"] == player) & (df["season"] == season)
        df.loc[mask, "period"] = df.loc[mask, "period"].map(period_map)
    return df


# df = collapse_short_periods(df, min_games=5)
# df["period_games"] = df.groupby(["player", "season", "period"])["game_count"].transform(
#     "size"
# )
# df = df.query("period_games >= 5 ")
# df["period"] = (
#     df.sort_values(by=["player", "season", "period"])
#     .groupby(["player"])["period"]
#     .rank(method="dense")
#     .astype(int)
# )

In [24]:
# from scipy.stats import zscore

# Calculate adjusted_bpm*min for each row (individual game)
df["adjusted_bpm*min"] = df["adjusted_bpm"] * df["minutes"]

# Calculate period-level adjusted_bpm_per_min: sum(bpm*min) / sum(min) for each player/season/period
period_stats = (
    df.groupby(["player", "season", "period"])
    .agg(
        period_bpm_min_sum=("adjusted_bpm*min", "sum"),
        period_min_sum=("minutes", "sum"),
    )
    .reset_index()
)
period_stats["adjusted_bpm_per_min"] = (
    period_stats["period_bpm_min_sum"] / period_stats["period_min_sum"]
)

# Merge period-level stat back to df
df = df.merge(
    period_stats[["player", "season", "period", "adjusted_bpm_per_min"]],
    on=["player", "season", "period"],
    how="left",
)

# Compute season mean and std for adjusted_bpm_per_min (over all periods in that season)
season_stats = (
    period_stats.groupby(["player", "season"])
    .agg(
        season_mean=("adjusted_bpm_per_min", "mean"),
        season_std=("adjusted_bpm_per_min", "std"),
    )
    .reset_index()
)

# Merge season stats back to df
df = df.merge(season_stats, on=["player", "season"], how="left")

# Calculate z-score for each period (all games in a period get the same z-score)
df["adjusted_bpm_per_min_zscore_by_period_season"] = (
    df["adjusted_bpm_per_min"] - df["season_mean"]
) / df["season_std"]

df

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,cumulative_adjusted_bpm_zscore,game_count,period,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season
0,401468968,2023,2,2023-02-06,2023-02-07T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,-1.161583,55,1,comeback,NaN,-199.432717,-0.477629,-0.477629,NaN,NaN
1,401469036,2023,2,2023-02-15,2023-02-16T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,1.270587,60,1,comeback,NaN,212.217083,-0.477629,-0.477629,NaN,NaN
2,401469365,2023,2,2023-04-07,2023-04-08T00:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,0.068767,81,1,comeback,NaN,16.879283,-0.477629,-0.477629,NaN,NaN
3,401469379,2023,2,2023-04-09,2023-04-09T19:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,-0.177771,82,1,comeback,NaN,-63.097697,-0.477629,-0.477629,NaN,NaN
4,401584915,2024,2,2023-12-02,2023-12-03T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,1.166410,19,2,comeback,NaN,109.178025,-2.261025,-2.261025,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311343,310408014,2011,2,2011-04-08,2011-04-08T23:30:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,0.603101,79,10,comeback,Other,91.912207,1.478978,1.110625,0.52093,0.707107
311344,310411001,2011,2,2011-04-11,2011-04-11T23:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,-0.317412,81,10,comeback,Other,-42.576298,1.478978,1.110625,0.52093,0.707107
311345,310413028,2011,2,2011-04-13,2011-04-14T00:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,0.151461,82,10,comeback,Other,33.027674,1.478978,1.110625,0.52093,0.707107
311346,401160820,2020,2,2019-11-16,2019-11-17T01:00:00Z,3137713.0,Zylan Cheatham,3,Pelicans,New Orleans,...,-0.707107,12,1,comeback,NaN,-207.838161,-6.593721,-6.593721,NaN,NaN


In [25]:
df.query("player == 'Stephen Curry' and season==2016")

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,cumulative_adjusted_bpm_zscore,game_count,period,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season
273951,400827890,2016,2,2015-10-27,2015-10-28T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.727227,1,13,season_start,NaN,937.611798,18.580763,18.580763,NaN,NaN
273952,400827917,2016,2,2015-10-30,2015-10-31T01:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.254163,2,13,season_start,NaN,577.642379,18.580763,18.580763,NaN,NaN
273953,400827922,2016,2,2015-10-31,2015-10-31T23:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,2.259192,3,13,season_start,NaN,1479.788718,18.580763,18.580763,NaN,NaN
273954,400827938,2016,2,2015-11-02,2015-11-03T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.852266,4,13,season_start,NaN,763.672341,18.580763,18.580763,NaN,NaN
273955,400827956,2016,2,2015-11-04,2015-11-05T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.263131,5,13,season_start,NaN,708.916962,18.580763,18.580763,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274025,400829050,2016,2,2016-04-05,2016-04-06T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-1.119296,78,13,season_start,NaN,339.353083,18.580763,18.580763,NaN,NaN
274026,400829064,2016,2,2016-04-07,2016-04-08T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-0.195238,79,13,season_start,NaN,611.142558,18.580763,18.580763,NaN,NaN
274027,400829077,2016,2,2016-04-09,2016-04-10T00:00:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-1.549009,80,13,season_start,NaN,124.695074,18.580763,18.580763,NaN,NaN
274028,400829088,2016,2,2016-04-10,2016-04-10T23:00:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.309222,81,13,season_start,NaN,767.740476,18.580763,18.580763,NaN,NaN


In [26]:
import pandas as pd
import numpy as np


def calculate_bpm_zscores_efficient(df):
    """
    Calculate BPM z-scores using minimal memory by processing one season at a time
    and avoiding unnecessary intermediate columns.

    Args:
        df: DataFrame with player stats including 'bpm', 'minutes', 'season', 'period', etc.

    Returns:
        Series with calculated z-scores that can be assigned back to the original dataframe
    """
    # Get unique seasons to process one at a time
    seasons = df["season"].unique()

    # Create an empty Series with the same index as the input dataframe
    result = pd.Series(index=df.index, dtype=float)

    for season in seasons:
        # Filter for the current season
        season_mask = df["season"] == season
        season_df = df.loc[season_mask, ["athlete_id", "bpm", "minutes", "period"]]

        # Step 1: Calculate weighted BPM for each player in the season
        # Group by player and get sum of minutes and weighted BPM
        player_stats = season_df.groupby("athlete_id").agg(
            {
                "minutes": "sum",
                "bpm": lambda x: np.average(
                    x, weights=season_df.loc[x.index, "minutes"]
                ),
            }
        )

        # Step 2: Calculate the season-wide weighted mean and std dev
        season_mean = np.average(player_stats["bpm"], weights=player_stats["minutes"])
        season_std = np.sqrt(
            np.average(
                (player_stats["bpm"] - season_mean) ** 2,
                weights=player_stats["minutes"],
            )
        )

        # Step 3: Calculate the weighted BPM for each period for each player
        # Process each unique athlete-period combination
        for athlete_id, period in season_df.groupby(
            ["athlete_id", "period"]
        ).groups.keys():
            # Get the rows for this player and period
            player_period_mask = (season_df["athlete_id"] == athlete_id) & (
                season_df["period"] == period
            )

            player_period_data = season_df.loc[player_period_mask]

            # Calculate weighted BPM for this period
            period_weighted_bpm = np.average(
                player_period_data["bpm"], weights=player_period_data["minutes"]
            )

            # Calculate z-score and store in result
            z_score = (period_weighted_bpm - season_mean) / season_std

            # Get the original indices for these rows
            original_indices = df.index[
                season_mask
                & (df["athlete_id"] == athlete_id)
                & (df["period"] == period)
            ]

            # Assign the z-score to all rows for this player-period in this season
            result.loc[original_indices] = z_score

    return result


# Example usage
df["bpm_zscore"] = calculate_bpm_zscores_efficient(df)
df

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,game_count,period,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
0,401468968,2023,2,2023-02-06,2023-02-07T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,55,1,comeback,NaN,-199.432717,-0.477629,-0.477629,NaN,NaN,-1.924328
1,401469036,2023,2,2023-02-15,2023-02-16T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,60,1,comeback,NaN,212.217083,-0.477629,-0.477629,NaN,NaN,-1.924328
2,401469365,2023,2,2023-04-07,2023-04-08T00:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,81,1,comeback,NaN,16.879283,-0.477629,-0.477629,NaN,NaN,-1.924328
3,401469379,2023,2,2023-04-09,2023-04-09T19:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,82,1,comeback,NaN,-63.097697,-0.477629,-0.477629,NaN,NaN,-1.924328
4,401584915,2024,2,2023-12-02,2023-12-03T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,19,2,comeback,NaN,109.178025,-2.261025,-2.261025,NaN,NaN,-1.378169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311343,310408014,2011,2,2011-04-08,2011-04-08T23:30:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,79,10,comeback,Other,91.912207,1.478978,1.110625,0.52093,0.707107,0.315754
311344,310411001,2011,2,2011-04-11,2011-04-11T23:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,81,10,comeback,Other,-42.576298,1.478978,1.110625,0.52093,0.707107,0.315754
311345,310413028,2011,2,2011-04-13,2011-04-14T00:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,82,10,comeback,Other,33.027674,1.478978,1.110625,0.52093,0.707107,0.315754
311346,401160820,2020,2,2019-11-16,2019-11-17T01:00:00Z,3137713.0,Zylan Cheatham,3,Pelicans,New Orleans,...,12,1,comeback,NaN,-207.838161,-6.593721,-6.593721,NaN,NaN,-2.959424


The best 10 periods of all time


In [27]:
df.sort_values(by="bpm_zscore", ascending=False).sort_values(
    by=["player", "season", "period"]
).drop_duplicates(subset=["player", "season", "period"], keep="last").sort_values(
    by="bpm_zscore"
).sort_values(
    by="bpm_zscore", ascending=False
).head(
    10
)

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,game_count,period,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
186535,401360849,2022,2,2022-03-15,2022-03-15T23:00:00Z,6442.0,Kyrie Irving,17,Nets,Brooklyn,...,69,51,comeback,Other,1436.423840,41.040681,11.682670,14.977923,1.960086,12.950721
25878,400828192,2016,2,2015-12-07,2015-12-08T00:00:00Z,4376.0,Boban Marjanovic,24,Spurs,San Antonio,...,22,8,comeback,Other,333.758037,19.632826,11.461517,5.869885,1.392073,8.495290
114061,400827938,2016,2,2015-11-02,2015-11-03T03:30:00Z,2489785.0,Ian Clark,9,Warriors,Golden State,...,4,9,comeback,Other,421.492406,35.124367,19.017226,22.778937,0.707107,8.288929
305404,401070891,2019,2,2018-11-14,2018-11-15T01:30:00Z,4032.0,Wesley Matthews,6,Mavericks,Dallas,...,14,19,comeback,Other,167.503988,7.976380,-2.846179,5.832743,1.855484,8.008510
5590,401224790,2020,2,2020-08-14,2020-08-15T01:00:00Z,6429.0,Alec Burks,20,76ers,Philadelphia,...,73,16,comeback,Foot,474.986816,21.590310,9.370089,17.282003,0.707107,7.943288
167009,400975379,2018,2,2018-01-13,2018-01-14T01:30:00Z,6450.0,Kawhi Leonard,24,Spurs,San Antonio,...,44,30,comeback,Shoulder,473.780466,16.920731,7.488497,9.658995,0.976523,7.862848
211212,400828192,2016,2,2015-12-07,2015-12-08T00:00:00Z,1996.0,Matt Bonner,24,Spurs,San Antonio,...,22,23,comeback,Other,276.393017,11.516376,1.797584,10.635744,0.913786,7.718071
204112,400975818,2018,2,2018-03-22,2018-03-22T23:00:00Z,2982329.0,Marcus Paige,30,Hornets,Charlotte,...,73,2,comeback,Other,124.625589,10.385466,10.385466,NaN,NaN,7.624659
13751,400828879,2016,2,2016-03-14,2016-03-15T00:00:00Z,6435.0,Andrew Goudelock,10,Rockets,Houston,...,67,2,comeback,NaN,207.406838,12.200402,12.200402,NaN,NaN,7.351191
139474,401361042,2022,2,2022-04-10,2022-04-10T23:00:00Z,4065648.0,Jayson Tatum,2,Celtics,Boston,...,82,16,comeback,Knee (Minor),593.378043,22.822232,11.300042,8.062772,1.429061,6.833074


In [28]:
df["period"] = df.groupby(["player"])["period"].rank(method="dense").astype(int)

In [29]:
curry = df.query("player == 'Stephen Curry'")
curry = curry.groupby("period").last()
curry

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,cumulative_adjusted_bpm_zscore,game_count,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
period,,,,,,,,,,,,,,,,,,,,,
1,300414022,2010,2,2010-04-14,2010-04-15T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,2.065900,82,season_start,None,823.435547,0.847524,0.847524,NaN,NaN,0.154052
2,301029009,2011,2,2010-10-29,2010-10-30T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.852172,2,season_start,None,247.039205,6.663947,4.044872,2.364946,1.107457,2.597402
3,301207006,2011,2,2010-12-07,2010-12-08T01:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-0.544567,21,comeback,Ankle,-95.650372,3.404888,4.044872,2.364946,-0.270613,0.206892
4,310413009,2011,2,2011-04-13,2011-04-14T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,1.327779,82,comeback,Ankle,312.975364,2.065780,4.044872,2.364946,-0.836844,0.489464
5,311226009,2012,2,2011-12-26,2011-12-27T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,1.083099,2,season_start,None,573.226977,1.992505,3.617619,1.701514,-0.955099,0.655226
6,320104024,2012,2,2012-01-04,2012-01-05T01:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.709182,6,comeback,Ankle,344.668530,3.473933,3.617619,1.701514,-0.084446,-0.845311
7,320310009,2012,2,2012-03-10,2012-03-11T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-0.003790,37,comeback,Ankle,61.678096,5.386420,3.617619,1.701514,1.039545,1.850212
8,400278379,2013,2,2013-01-28,2013-01-29T00:00:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,0.329989,44,season_start,None,203.163546,3.919685,5.241758,1.869694,-0.707107,1.720343
9,400278950,2013,2,2013-04-17,2013-04-18T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,-0.000159,81,comeback,Other,154.437522,6.563832,5.241758,1.869694,0.707107,2.512444


In [30]:
injuries.query(
    "Acquired == 'Stephen Curry' or Relinquished == 'Stephen Curry'"
).sort_values("Date").head(20)

,Date,Team,Acquired,Relinquished,Notes,Injury_Category
10837,2010-10-31,Warriors,NaN,Stephen Curry,placed on IL with sprained right ankle,Ankle
10879,2010-11-05,Warriors,Stephen Curry,NaN,activated from IL,Ankle
11231,2010-12-11,Warriors,NaN,Stephen Curry,placed on IL with sprained left ankle,Ankle
11391,2010-12-25,Warriors,Stephen Curry,NaN,activated from IL,Ankle
12561,2011-12-28,Warriors,NaN,Stephen Curry,placed on IL with sprained right ankle,Ankle
12572,2011-12-31,Warriors,Stephen Curry,NaN,activated from IL,Ankle
12637,2012-01-06,Warriors,NaN,Stephen Curry,placed on IL with sprained right ankle,Ankle
12741,2012-01-20,Warriors,Stephen Curry,NaN,activated from IL,Ankle
13240,2012-03-19,Warriors,NaN,Stephen Curry,placed on IL (F),Other
13304,2012-03-27,Warriors,NaN,Stephen Curry,placed on IL with sprained right ankle (out fo...,Ankle


In [31]:
curry["next_period_zscore"] = curry["bpm_zscore"].shift(-1)
curry

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,game_count,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore,next_period_zscore
period,,,,,,,,,,,,,,,,,,,,,
1,300414022,2010,2,2010-04-14,2010-04-15T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,82,season_start,None,823.435547,0.847524,0.847524,NaN,NaN,0.154052,2.597402
2,301029009,2011,2,2010-10-29,2010-10-30T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,2,season_start,None,247.039205,6.663947,4.044872,2.364946,1.107457,2.597402,0.206892
3,301207006,2011,2,2010-12-07,2010-12-08T01:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,21,comeback,Ankle,-95.650372,3.404888,4.044872,2.364946,-0.270613,0.206892,0.489464
4,310413009,2011,2,2011-04-13,2011-04-14T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,82,comeback,Ankle,312.975364,2.065780,4.044872,2.364946,-0.836844,0.489464,0.655226
5,311226009,2012,2,2011-12-26,2011-12-27T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,2,season_start,None,573.226977,1.992505,3.617619,1.701514,-0.955099,0.655226,-0.845311
6,320104024,2012,2,2012-01-04,2012-01-05T01:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,6,comeback,Ankle,344.668530,3.473933,3.617619,1.701514,-0.084446,-0.845311,1.850212
7,320310009,2012,2,2012-03-10,2012-03-11T03:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,37,comeback,Ankle,61.678096,5.386420,3.617619,1.701514,1.039545,1.850212,1.720343
8,400278379,2013,2,2013-01-28,2013-01-29T00:00:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,44,season_start,None,203.163546,3.919685,5.241758,1.869694,-0.707107,1.720343,2.512444
9,400278950,2013,2,2013-04-17,2013-04-18T02:30:00Z,3975.0,Stephen Curry,9,Warriors,Golden State,...,81,comeback,Other,154.437522,6.563832,5.241758,1.869694,0.707107,2.512444,2.221026


In [32]:
# df.to_csv("./data/df_with_bpm_zscores.csv", index=False)

In [33]:
df = pd.read_csv("./data/df_with_bpm_zscores.csv", dtype={56: str, 149: str})
df

,game_id,season,season_type,game_date,game_date_time,athlete_id,player,team_id,team_name,team_location,...,game_count,period,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
0,401468968,2023,2,2023-02-06,2023-02-07T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,55,1,comeback,NaN,-199.432717,-0.477629,-0.477629,NaN,NaN,-1.924328
1,401469036,2023,2,2023-02-15,2023-02-16T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,60,1,comeback,NaN,212.217083,-0.477629,-0.477629,NaN,NaN,-1.924328
2,401469365,2023,2,2023-04-07,2023-04-08T00:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,81,1,comeback,NaN,16.879283,-0.477629,-0.477629,NaN,NaN,-1.924328
3,401469379,2023,2,2023-04-09,2023-04-09T19:30:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,82,1,comeback,NaN,-63.097697,-0.477629,-0.477629,NaN,NaN,-1.924328
4,401584915,2024,2,2023-12-02,2023-12-03T02:00:00Z,4067017.0,A.J. Lawson,6,Mavericks,Dallas,...,19,2,comeback,NaN,109.178025,-2.261025,-2.261025,NaN,NaN,-1.378169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311343,310408014,2011,2,2011-04-08,2011-04-08T23:30:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,79,3,comeback,Other,91.912207,1.478978,1.110625,0.52093,0.707107,0.315754
311344,310411001,2011,2,2011-04-11,2011-04-11T23:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,81,3,comeback,Other,-42.576298,1.478978,1.110625,0.52093,0.707107,0.315754
311345,310413028,2011,2,2011-04-13,2011-04-14T00:00:00Z,362.0,Zydrunas Ilgauskas,14,Heat,Miami,...,82,3,comeback,Other,33.027674,1.478978,1.110625,0.52093,0.707107,0.315754
311346,401160820,2020,2,2019-11-16,2019-11-17T01:00:00Z,3137713.0,Zylan Cheatham,3,Pelicans,New Orleans,...,12,1,comeback,NaN,-207.838161,-6.593721,-6.593721,NaN,NaN,-2.959424


In [34]:
model_df = df.groupby(["player", "period"]).last().reset_index()

model_df

,player,period,game_id,season,season_type,game_date,game_date_time,athlete_id,team_id,team_name,...,cumulative_adjusted_bpm_zscore,game_count,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
0,A.J. Lawson,1,401469379,2023,2,2023-04-09,2023-04-09T19:30:00Z,4067017.0,6,Mavericks,...,-0.177771,82,comeback,None,-63.097697,-0.477629,-0.477629,NaN,NaN,-1.924328
1,A.J. Lawson,2,401585824,2024,2,2024-04-14,2024-04-14T19:30:00Z,4067017.0,6,Mavericks,...,-0.327482,82,comeback,None,-145.930756,-2.261025,-2.261025,NaN,NaN,-1.378169
2,A.J. Price,1,300412011,2010,2,2010-04-12,2010-04-12T23:00:00Z,4010.0,11,Pacers,...,1.885894,81,comeback,Other,302.986407,-3.192203,-3.192203,NaN,NaN,-0.603327
3,A.J. Price,2,301120011,2011,2,2010-11-20,2010-11-21T00:00:00Z,4010.0,11,Pacers,...,-0.370254,11,comeback,Other,-116.286562,2.413020,-1.766865,5.91125,0.707107,2.737438
4,A.J. Price,3,310413019,2011,2,2011-04-13,2011-04-14T00:00:00Z,4010.0,11,Pacers,...,1.228125,82,comeback,Other,87.038427,-5.946750,-1.766865,5.91125,-0.707107,-0.786252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13444,Zoran Dragic,1,400579519,2015,2,2015-04-15,2015-04-16T00:00:00Z,2560823.0,14,Heat,...,0.000000,82,comeback,Other,-19.205061,-0.468416,-0.468416,NaN,NaN,0.546422
13445,Zydrunas Ilgauskas,1,300414001,2010,2,2010-04-14,2010-04-15T00:00:00Z,362.0,5,Cavaliers,...,-0.395681,82,season_start,None,-100.457927,-0.975379,-0.975379,NaN,NaN,-0.688724
13446,Zydrunas Ilgauskas,2,310310014,2011,2,2011-03-10,2011-03-11T00:00:00Z,362.0,14,Heat,...,-0.319498,65,comeback,Other,-48.780907,0.742271,1.110625,0.52093,-0.707107,-0.119979
13447,Zydrunas Ilgauskas,3,310413028,2011,2,2011-04-13,2011-04-14T00:00:00Z,362.0,14,Heat,...,0.151461,82,comeback,Other,33.027674,1.478978,1.110625,0.52093,0.707107,0.315754


In [35]:
model_df["previous_period_zscore"] = model_df["bpm_zscore"].shift(1)
model_df["next_player"] = model_df["player"].shift(-1)
model_df.loc[
    model_df["player"] != model_df["next_player"], "previous_period_zscore"
] = float("nan")
model_df["difference"] = -model_df["previous_period_zscore"] + model_df["bpm_zscore"]
model_df.drop(columns="next_player", inplace=True)
model_df[
    [
        "player",
        "period",
        "bpm_zscore",
        "previous_period_zscore",
        "difference",
        "comeback_from_injury",
    ]
].head(10)

,player,period,bpm_zscore,previous_period_zscore,difference,comeback_from_injury
0,A.J. Lawson,1,-1.924328,NaN,NaN,None
1,A.J. Lawson,2,-1.378169,NaN,NaN,None
2,A.J. Price,1,-0.603327,-1.378169,0.774842,Other
3,A.J. Price,2,2.737438,-0.603327,3.340765,Other
4,A.J. Price,3,-0.786252,2.737438,-3.523690,Other
5,A.J. Price,4,0.497503,-0.786252,1.283755,Other
6,A.J. Price,5,-0.256082,0.497503,-0.753585,None
7,A.J. Price,6,-0.382755,-0.256082,-0.126674,Other
8,A.J. Price,7,0.447293,-0.382755,0.830049,Other
9,A.J. Price,8,0.203389,0.447293,-0.243904,Other


In [36]:
model_df.query("comeback_from_injury == 'ACL Tear'").sort_values(by="game_date").head(
    20
)

,player,period,game_id,season,season_type,game_date,game_date_time,athlete_id,team_id,team_name,...,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore,previous_period_zscore,difference
9450,Michael Redd,2,291127025,2010,2,2009-11-27,2009-11-28T02:30:00Z,692.0,15,Bucks,...,comeback,ACL Tear,-131.718929,-7.549011,-4.889036,2.784730,-0.955200,-2.146614,0.670358,-2.816971
9553,Mike Wilks,1,291204025,2010,2,2009-12-04,2009-12-05T01:00:00Z,1913.0,25,Thunder,...,comeback,ACL Tear,-188.328217,-7.199676,-7.199676,NaN,NaN,-2.420886,NaN,NaN
5913,Jason Smith,2,300407014,2010,2,2010-04-07,2010-04-07T23:30:00Z,3232.0,20,76ers,...,comeback,ACL Tear,-251.952602,-6.508999,-3.912031,3.672667,-0.707107,-2.182822,-1.565053,-0.617769
187,Al Jefferson,2,300414016,2010,2,2010-04-14,2010-04-15T00:00:00Z,2389.0,16,Timberwolves,...,comeback,ACL Tear,75.919746,-2.626899,-3.494145,1.226471,0.707107,-1.094191,-0.912778,-0.181414
6955,Josh Howard,6,301231011,2011,2,2010-12-31,2010-12-31T20:00:00Z,2006.0,27,Wizards,...,comeback,ACL Tear,-149.634661,-6.898492,-9.806380,4.677538,0.621671,-0.463238,-0.826289,0.363051
9452,Michael Redd,4,310413025,2011,2,2011-04-13,2011-04-14T00:00:00Z,692.0,15,Bucks,...,comeback,ACL Tear,-222.252426,-9.814367,-9.814367,NaN,NaN,-1.170096,0.058084,-1.228180
4145,Eric Maynor,4,400278557,2013,2,2013-02-24,2013-02-25T02:00:00Z,4001.0,22,Trail Blazers,...,comeback,ACL Tear,-232.718942,1.025688,-1.882902,4.113368,0.707107,0.908844,-0.035741,0.944585
2799,David West,6,400278701,2013,2,2013-03-16,2013-03-16T23:30:00Z,2177.0,11,Pacers,...,comeback,ACL Tear,-194.734006,3.554443,1.430015,1.963304,1.082068,2.060935,0.872440,1.188495
10881,Ricky Rubio,2,400278945,2013,2,2013-04-17,2013-04-18T00:00:00Z,4011.0,16,Timberwolves,...,comeback,ACL Tear,-101.346444,-1.392258,-1.392258,NaN,NaN,0.261320,0.807098,-0.545778
1383,Brandon Rush,6,400489649,2014,2,2014-02-11,2014-02-12T03:30:00Z,3457.0,26,Jazz,...,comeback,ACL Tear,-26.720570,-5.750931,-5.750931,NaN,NaN,-1.924942,0.374684,-2.299627


In [37]:
model_df.query("player == 'Kevin Durant'")[
    [
        "player",
        "period",
        "season",
        "bpm_zscore",
        "previous_period_zscore",
        "difference",
        "comeback_from_injury",
    ]
]

,player,period,season,bpm_zscore,previous_period_zscore,difference,comeback_from_injury
7568,Kevin Durant,1,2010,2.443927,-0.643698,3.087625,None
7569,Kevin Durant,2,2011,0.429896,2.443927,-2.014031,None
7570,Kevin Durant,3,2011,0.655040,0.429896,0.225144,Ankle
7571,Kevin Durant,4,2011,2.323267,0.655040,1.668228,Knee (Minor)
7572,Kevin Durant,5,2012,2.470466,2.323267,0.147198,None
7573,Kevin Durant,6,2013,3.570419,2.470466,1.099953,None
7574,Kevin Durant,7,2014,3.569572,3.570419,-0.000847,None
7575,Kevin Durant,8,2015,2.464975,3.569572,-1.104597,Other
7576,Kevin Durant,9,2015,2.340292,2.464975,-0.124683,Foot
7577,Kevin Durant,10,2015,-1.840481,2.340292,-4.180773,Foot


In [38]:
model_df.query("comeback_from_injury == 'Achilles (Tear)'")

,player,period,game_id,season,season_type,game_date,game_date_time,athlete_id,team_id,team_name,...,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore,previous_period_zscore,difference
1316,Brandon Clarke,16,401585779,2024,2,2024-04-09,2024-04-10T00:00:00Z,3906665.0,29,Grizzlies,...,comeback,Achilles (Tear),-223.651733,-5.084641,-5.084641,NaN,NaN,-1.316551,NaN,NaN
1351,Brandon Jennings,9,400829053,2016,2,2016-04-06,2016-04-06T23:00:00Z,3997.0,19,Magic,...,comeback,Achilles (Tear),-78.895514,1.473724,1.473724,NaN,NaN,-0.213525,0.952398,-1.165922
1863,Chauncey Billups,9,400277969,2013,2,2012-12-03,2012-12-04T02:00:00Z,63.0,12,Clippers,...,comeback,Achilles (Tear),-90.938492,1.193442,9.182466,12.947441,-0.617035,1.075377,0.488895,0.586482
3848,Dwight Powell,10,401307375,2021,2,2021-03-13,2021-03-14T03:00:00Z,2531367.0,6,Mavericks,...,comeback,Achilles (Tear),41.577517,-1.120673,2.101669,4.569408,-0.705199,-1.551884,-0.248652,-1.303232
6690,Jonas Jerebko,4,400490093,2014,2,2014-04-16,2014-04-17T00:00:00Z,3998.0,8,Pistons,...,comeback,Achilles (Tear),-209.235393,-0.932797,-0.932797,NaN,NaN,-1.465883,-1.248908,-0.216975
7596,Kevin Durant,29,401267505,2021,2,2021-02-05,2021-02-06T00:30:00Z,3202.0,17,Nets,...,comeback,Achilles (Tear),76.625994,12.894143,11.320856,5.542688,0.283849,1.655484,2.356267,-0.700783
7863,Klay Thompson,15,401360502,2022,2,2022-01-20,2022-01-21T03:00:00Z,6475.0,9,Warriors,...,comeback,Achilles (Tear),-359.655976,-1.855837,3.180034,7.846115,-0.641830,-0.923667,0.704472,-1.628139
7885,Kobe Bryant,6,400489239,2014,2,2013-12-17,2013-12-18T01:00:00Z,110.0,13,Lakers,...,comeback,Achilles (Tear),26.127822,-4.458133,-4.458133,NaN,NaN,-1.548616,1.528587,-3.077202
9325,Mehmet Okur,2,301218015,2011,2,2010-12-18,2010-12-19T01:30:00Z,1014.0,26,Jazz,...,comeback,Achilles (Tear),-175.745201,-13.518862,-4.228310,8.737475,-1.063299,-1.586333,0.283957,-1.870290
11017,Rodney Hood,25,401267564,2021,2,2021-02-12,2021-02-13T03:00:00Z,2581177.0,22,Trail Blazers,...,comeback,Achilles (Tear),-181.112647,-3.970684,-6.421039,3.213506,0.762518,-1.097400,-2.260794,1.163395


In [39]:
model_df = model_df.dropna(subset=["difference", "bpm_zscore"])
model_df = model_df[["difference", "comeback_from_injury"]]
model_df["comeback_from_injury"] = model_df["comeback_from_injury"].fillna("None")
model_df = pd.get_dummies(
    model_df, columns=["comeback_from_injury"], drop_first=False, dtype=int
)
model_df

,difference,comeback_from_injury_ACL Tear,comeback_from_injury_Achilles (Minor),comeback_from_injury_Achilles (Tear),comeback_from_injury_Ankle,comeback_from_injury_Back,comeback_from_injury_Calf,comeback_from_injury_Concussion,comeback_from_injury_Elbow,comeback_from_injury_Eye,...,comeback_from_injury_Illness/Medical,comeback_from_injury_Knee (Major),comeback_from_injury_Knee (Minor),comeback_from_injury_Leg,comeback_from_injury_None,comeback_from_injury_Other,comeback_from_injury_Quad/Thigh,comeback_from_injury_Shoulder,comeback_from_injury_Surgery,comeback_from_injury_Wrist
2,0.774842,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,3.340765,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,-3.523690,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
5,1.283755,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
6,-0.753585,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13440,0.169712,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13441,1.086899,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13442,-1.734223,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
13445,-1.235146,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


Note that comeback_from_injury is the injury the player is coming back from, so if they got injured before and they cameback, the injury they had is what goes in comeback_from_injury. So, we can predict the regression based on this. given their previous period, and the fact that they had x injury this period, what should their z score difference be across periods.


Should we instead make a new model for each and fit the current score on the next, then calculate r score like that. since they shouldn't be correlated together


In [40]:
kd = (
    injuries.sort_values(by=["Relinquished", "Date"])
    .query("Acquired == 'Kevin Durant' or Relinquished == 'Kevin Durant'")
    .sort_values("Date")
)

In [42]:
# kd = injuries_melted.query('player == "Kevin Durant"').sort_values("Date")

In [43]:
df.groupby(["player", "period"]).last()["comeback_from_injury"].value_counts()

comeback_from_injury
Other                    2696
Ankle                    1041
Knee (Minor)              994
Illness/Medical           538
Back                      443
Foot                      405
Hamstring                 266
Hip                       247
Calf                      221
Shoulder                  206
Quad/Thigh                174
Groin                     170
Finger                    153
Concussion                138
Wrist                     111
Achilles (Minor)           95
Hand                       85
General Strain/Sprain      80
Leg                        72
Elbow                      64
Head/Neck                  58
Knee (Major)               42
Surgery                    34
ACL Tear                   33
Eye                        25
Achilles (Tear)            11
Name: count, dtype: int64

In [44]:
df.groupby(["player", "period"]).last().query(
    "comeback_from_injury == 'Achilles (Tear)'"
)

,,game_id,season,season_type,game_date,game_date_time,athlete_id,team_id,team_name,team_location,team_short_display_name,...,cumulative_adjusted_bpm_zscore,game_count,period_event_type,comeback_from_injury,adjusted_bpm*min,adjusted_bpm_per_min,season_mean,season_std,adjusted_bpm_per_min_zscore_by_period_season,bpm_zscore
player,period,,,,,,,,,,,,,,,,,,,,,
Brandon Clarke,16,401585779,2024,2,2024-04-09,2024-04-10T00:00:00Z,3906665.0,29,Grizzlies,Memphis,Grizzlies,...,-0.392268,79,comeback,Achilles (Tear),-223.651733,-5.084641,-5.084641,NaN,NaN,-1.316551
Brandon Jennings,9,400829053,2016,2,2016-04-06,2016-04-06T23:00:00Z,3997.0,19,Magic,Orlando,Magic,...,-0.576428,78,comeback,Achilles (Tear),-78.895514,1.473724,1.473724,NaN,NaN,-0.213525
Chauncey Billups,9,400277969,2013,2,2012-12-03,2012-12-04T02:00:00Z,63.0,12,Clippers,LA,Clippers,...,-0.574716,17,comeback,Achilles (Tear),-90.938492,1.193442,9.182466,12.947441,-0.617035,1.075377
Dwight Powell,10,401307375,2021,2,2021-03-13,2021-03-14T03:00:00Z,2531367.0,6,Mavericks,Dallas,Mavericks,...,-0.177100,37,comeback,Achilles (Tear),41.577517,-1.120673,2.101669,4.569408,-0.705199,-1.551884
Jonas Jerebko,4,400490093,2014,2,2014-04-16,2014-04-17T00:00:00Z,3998.0,8,Pistons,Detroit,Pistons,...,-0.823843,82,comeback,Achilles (Tear),-209.235393,-0.932797,-0.932797,NaN,NaN,-1.465883
Kevin Durant,29,401267505,2021,2,2021-02-05,2021-02-06T00:30:00Z,3202.0,17,Nets,Brooklyn,Nets,...,-1.286129,24,comeback,Achilles (Tear),76.625994,12.894143,11.320856,5.542688,0.283849,1.655484
Klay Thompson,15,401360502,2022,2,2022-01-20,2022-01-21T03:00:00Z,6475.0,9,Warriors,Golden State,Warriors,...,-1.683517,45,comeback,Achilles (Tear),-359.655976,-1.855837,3.180034,7.846115,-0.641830,-0.923667
Kobe Bryant,6,400489239,2014,2,2013-12-17,2013-12-18T01:00:00Z,110.0,13,Lakers,Los Angeles,Lakers,...,0.722022,25,comeback,Achilles (Tear),26.127822,-4.458133,-4.458133,NaN,NaN,-1.548616
Mehmet Okur,2,301218015,2011,2,2010-12-18,2010-12-19T01:30:00Z,1014.0,26,Jazz,Utah,Jazz,...,-1.507133,28,comeback,Achilles (Tear),-175.745201,-13.518862,-4.228310,8.737475,-1.063299,-1.586333


In [45]:
# def collapse_short_periods(df, min_games=5):
#     df = df.copy()
#     # For each player/season, process periods
#     for (player, season), group in df.groupby(["player", "season"]):
#         period_counts = group["period"].value_counts().sort_index()
#         periods = period_counts.index.tolist()
#         counts = period_counts.values
#         # Map for period reassignment
#         period_map = {}
#         prev_period = None
#         for i, (p, count) in enumerate(zip(periods, counts)):
#             if count < min_games and prev_period is not None:
#                 period_map[p] = prev_period
#             else:
#                 period_map[p] = p
#                 prev_period = p
#         mask = (df["player"] == player) & (df["season"] == season)
#         df.loc[mask, "period"] = df.loc[mask, "period"].map(period_map)
#     return df


# df = collapse_short_periods(df, min_games=5)

In [54]:
model_df = df.groupby(["player", "period"]).last().reset_index()
model_df = model_df.sort_values(by=["player", "season", "period"])
model_df["previous_period_zscore"] = model_df["bpm_zscore"].shift(1)
model_df["next_player"] = model_df["player"].shift(-1)
model_df.loc[
    model_df["player"] != model_df["next_player"], "previous_period_zscore"
] = float("nan")
model_df["difference"] = -model_df["previous_period_zscore"] + model_df["bpm_zscore"]
model_df.drop(columns="next_player", inplace=True)
model_df["comeback_from_injury"] = model_df["comeback_from_injury"].fillna("None")
model_df = model_df[
    ["previous_period_zscore", "bpm_zscore", "difference", "comeback_from_injury"]
]
model_df = model_df.dropna().reset_index(drop=True)
model_df

,previous_period_zscore,bpm_zscore,difference,comeback_from_injury
0,-1.378169,-0.603327,0.774842,Other
1,-0.603327,2.737438,3.340765,Other
2,2.737438,-0.786252,-3.523690,Other
3,-0.786252,0.497503,1.283755,Other
4,0.497503,-0.256082,-0.753585,None
...,...,...,...,...
11815,1.223389,1.393102,0.169712,Finger
11816,1.393102,2.480000,1.086899,Foot
11817,2.480000,0.745777,-1.734223,Other
11818,0.546422,-0.688724,-1.235146,None


In [56]:
model_df.groupby("comeback_from_injury")["difference"].describe().sort_values(
    by="mean", ascending=True
)

,count,mean,std,min,25%,50%,75%,max
comeback_from_injury,,,,,,,,
Achilles (Tear),10.0,-0.786281,1.284585,-3.077202,-1.546912,-0.933353,0.208151,1.163395
ACL Tear,29.0,-0.514696,1.125006,-2.844481,-1.228180,-0.578800,0.553568,1.251275
Knee (Major),38.0,-0.387532,1.464723,-4.284354,-0.987239,-0.504572,0.452206,3.334223
Calf,201.0,-0.197373,1.541654,-7.108788,-0.958771,-0.140777,0.612751,3.712637
Back,407.0,-0.157563,1.631775,-5.900802,-1.078578,-0.043897,0.805456,4.975013
Groin,158.0,-0.148598,1.640837,-5.223197,-0.981394,-0.125468,0.750285,4.373717
Hamstring,246.0,-0.108030,1.694331,-8.223710,-1.084446,0.019910,0.827980,5.240674
Quad/Thigh,168.0,-0.107019,1.502057,-5.083633,-0.893801,-0.040781,0.661975,6.047410
Leg,64.0,-0.078599,1.468425,-4.683679,-0.776028,-0.144250,0.530605,4.970555


In [57]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

results = []

# Assume 'difference', 'previous_period_zscore', 'injury_type' are in your DataFrame
for injury in model_df["comeback_from_injury"].dropna().unique():
    subset = model_df[model_df["comeback_from_injury"] == injury]

    # X = previous z-score, y = difference
    X = subset[["previous_period_zscore"]]
    y = subset["difference"]

    # Drop rows where X or y is NaN or inf
    valid = (
        X.notnull().all(axis=1)
        & y.notnull()
        & np.isfinite(X).all(axis=1)
        & np.isfinite(y)
    )
    X = X[valid]
    y = y[valid]

    X = sm.add_constant(X)  # Adds intercept

    if len(X) == 0:
        continue  # Skip if no valid data

    model = sm.OLS(y, X).fit()

    r2 = model.rsquared
    rmse = np.sqrt(mean_squared_error(y, model.predict(X)))
    coef = model.params["previous_period_zscore"]
    intercept = model.params["const"]

    results.append(
        {
            "comeback_from_injury": injury,
            "r2": r2,
            "rmse": rmse,
            "coef": coef,
            "intercept": intercept,
            "n_samples": len(subset),
        }
    )

# Convert to DataFrame for viewing
results_df = pd.DataFrame(results).sort_values("r2", ascending=False)
results_df

,comeback_from_injury,r2,rmse,coef,intercept,n_samples
25,Surgery,0.622374,1.344800,-0.969473,-0.317099,30
19,Knee (Major),0.546914,0.972870,-0.817643,-0.537148,38
12,General Strain/Sprain,0.455203,1.420482,-0.691849,-0.052942,71
1,None,0.429422,1.138712,-0.693597,-0.104884,4357
21,Leg,0.405229,1.123587,-0.679898,-0.237935,64
26,Eye,0.382354,1.081197,-0.745718,0.009219,24
9,Achilles (Minor),0.347445,1.624762,-0.760210,0.002810,88
0,Other,0.344546,1.749637,-0.701338,-0.392297,2239
2,Ankle,0.319282,1.343609,-0.608880,-0.172426,974
24,Achilles (Tear),0.313870,1.009456,-0.546305,-0.629834,10


In [58]:
injuries.to_csv("./data/injuries_code_classified_all_categories.csv", index=False)